# HW03

姓名：邓烨涛  
学号：20234080108




In [1]:
import math
import random
import numpy as np
import torch
from torch import nn
from torchvision import transforms

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)


cpu


## 2 卷积层

### 2.1 理论题

题目给出输入张量大小为 $3\times 32\times 32$，即通道数为 3，高和宽都是 32。卷积核共有 16 个，每个卷积核大小为 $3\times 5\times 5$，padding 为 2，stride 为 2。

卷积输出空间尺寸计算公式为

$$
H_{out}=\left\lfloor\frac{H+2P-K}{S}\right\rfloor+1,\quad
W_{out}=\left\lfloor\frac{W+2P-K}{S}\right\rfloor+1.
$$

代入 $H=W=32$，$P=2$，$K=5$，$S=2$：

$$
H_{out}=W_{out}=\left\lfloor\frac{32+2\times 2-5}{2}\right\rfloor+1
=\left\lfloor\frac{31}{2}\right\rfloor+1=16.
$$

因为卷积核数量是 16，所以输出特征图的通道数也是 16。最终输出 Feature Map 大小为：

$$
16\times 16\times 16.
$$

一次卷积的具体操作是：对每个输出位置，取输入中对应的 $3\times 5\times 5$ 局部区域，与一个同样大小的卷积核逐元素相乘后求和，再加上偏置项。由于有 16 个卷积核，所以每个空间位置会得到 16 个输出值。


### 2.2 编程题：手写 Max Pooling 前向计算

下面不用 `torch.nn.MaxPool2d`，而是手动实现二维最大池化。函数支持输入为 `(N, C, H, W)` 的张量，并支持 `kernel_size`、`stride` 和 `padding`。


In [2]:
def _pair(x):
    if isinstance(x, tuple):
        return x
    return (x, x)


def manual_max_pool2d(X, kernel_size, stride=None, padding=0):
    kernel_h, kernel_w = _pair(kernel_size)
    if stride is None:
        stride = kernel_size
    stride_h, stride_w = _pair(stride)
    pad_h, pad_w = _pair(padding)

    if X.dim() != 4:
        raise ValueError("X should have shape (N, C, H, W)")

    if pad_h > 0 or pad_w > 0:
        X = torch.nn.functional.pad(
            X,
            (pad_w, pad_w, pad_h, pad_h),
            mode="constant",
            value=float("-inf"),
        )

    N, C, H, W = X.shape
    out_h = (H - kernel_h) // stride_h + 1
    out_w = (W - kernel_w) // stride_w + 1
    Y = torch.empty((N, C, out_h, out_w), dtype=X.dtype, device=X.device)

    for i in range(out_h):
        for j in range(out_w):
            h0 = i * stride_h
            w0 = j * stride_w
            window = X[:, :, h0:h0 + kernel_h, w0:w0 + kernel_w]
            Y[:, :, i, j] = window.amax(dim=(2, 3))
    return Y


X = torch.tensor([[[[1., 2., 3., 4.],
                    [5., 6., 7., 8.],
                    [9., 10., 11., 12.],
                    [13., 14., 15., 16.]]]])

manual_y = manual_max_pool2d(X, kernel_size=2, stride=2)
torch_y = nn.MaxPool2d(kernel_size=2, stride=2)(X)
print(manual_y)
print("same as torch:", torch.allclose(manual_y, torch_y))


tensor([[[[ 6.,  8.],
          [14., 16.]]]])
same as torch: True


## 3 LeNet、AlexNet、VGG 和 NiN

### 3.1 理论题

VGG 中常用多个 $3\times 3$ 卷积层代替一个更大的卷积核。假设输入和输出通道数都为 $C$，并且暂时不考虑偏置项。

**1. 一个 $5\times 5$ 卷积层的参数量**

一个 $5\times 5$ 卷积层的参数量为

$$
5\times 5\times C\times C=25C^2.
$$

如果计算偏置，还需要再加 $C$ 个偏置参数。

**2. 两个 $3\times 3$ 卷积层的参数量**

一个 $3\times 3$ 卷积层参数量为 $9C^2$，两个这样的卷积层总参数量为

$$
2\times 3\times 3\times C\times C=18C^2.
$$

如果计算偏置，则为 $18C^2+2C$。

两个 $3\times 3$ 卷积层的感受野等价于一个 $5\times 5$ 卷积层，但参数量从 $25C^2$ 降到 $18C^2$，中间还多了一次非线性激活，因此表达能力通常更强。


### 3.2 编程题：实现 NiN Block

NiN 的主要想法是在普通卷积后接若干个 $1\times 1$ 卷积，相当于在每个像素位置上使用小型多层感知机，从而增强通道维度上的非线性表达能力。


In [3]:
def nin_block(in_channels, out_channels, kernel_size, stride, padding):
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding),
        nn.ReLU(),
        nn.Conv2d(out_channels, out_channels, kernel_size=1),
        nn.ReLU(),
        nn.Conv2d(out_channels, out_channels, kernel_size=1),
        nn.ReLU(),
    )


block = nin_block(3, 16, kernel_size=5, stride=1, padding=2)
X = torch.randn(2, 3, 32, 32)
Y = block(X)
print(Y.shape)


torch.Size([2, 16, 32, 32])


## 4 Inception 与残差网络

### 4.1 理论题：Batch Normalization 计算

mini-batch 中 4 个特征值为：

$$
x_1=2,\quad x_2=4,\quad x_3=6,\quad x_4=8.
$$

Batch Normalization 先计算均值：

$$
\mu=\frac{2+4+6+8}{4}=5.
$$

方差为：

$$
\sigma^2=\frac{(2-5)^2+(4-5)^2+(6-5)^2+(8-5)^2}{4}
=\frac{9+1+1+9}{4}=5.
$$

题目给出 $\gamma=2$，$\beta=1$，$\epsilon=0$。所以

$$
\hat{x}_i=\frac{x_i-\mu}{\sqrt{\sigma^2+\epsilon}}
=\frac{x_i-5}{\sqrt{5}},
$$

最终输出为

$$
y_i=\gamma\hat{x}_i+\beta=2\frac{x_i-5}{\sqrt{5}}+1.
$$

于是：

$$
y_1=1-\frac{6}{\sqrt5}\approx -1.6833,
$$

$$
y_2=1-\frac{2}{\sqrt5}\approx 0.1056,
$$

$$
y_3=1+\frac{2}{\sqrt5}\approx 1.8944,
$$

$$
y_4=1+\frac{6}{\sqrt5}\approx 3.6833.
$$


In [4]:
x = torch.tensor([2., 4., 6., 8.])
gamma = 2.0
beta = 1.0
eps = 0.0
mu = x.mean()
var = ((x - mu) ** 2).mean()
y = gamma * (x - mu) / torch.sqrt(var + eps) + beta
print("mean:", mu.item())
print("var:", var.item())
print("BN output:", y)


mean: 5.0
var: 5.0
BN output: tensor([-1.6833,  0.1056,  1.8944,  3.6833])


### 4.2 编程题：实现 ResNet 残差块

残差网络通过学习 $f(x)+x$ 缓解深层网络训练困难。如果输入和输出通道数或空间尺寸不同，可以用 $1\times 1$ 卷积对输入 $x$ 做投影，使它能与主分支输出相加。


In [5]:
class Residual(nn.Module):
    def __init__(self, input_channels, num_channels, use_1x1conv=False, strides=1):
        super().__init__()
        self.conv1 = nn.Conv2d(
            input_channels,
            num_channels,
            kernel_size=3,
            padding=1,
            stride=strides,
        )
        self.conv2 = nn.Conv2d(
            num_channels,
            num_channels,
            kernel_size=3,
            padding=1,
        )
        if use_1x1conv:
            self.conv3 = nn.Conv2d(
                input_channels,
                num_channels,
                kernel_size=1,
                stride=strides,
            )
        else:
            self.conv3 = None
        self.bn1 = nn.BatchNorm2d(num_channels)
        self.bn2 = nn.BatchNorm2d(num_channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, X):
        Y = self.relu(self.bn1(self.conv1(X)))
        Y = self.bn2(self.conv2(Y))
        if self.conv3 is not None:
            X = self.conv3(X)
        Y += X
        return self.relu(Y)


blk_same = Residual(3, 3)
print(blk_same(torch.randn(4, 3, 32, 32)).shape)

blk_down = Residual(3, 16, use_1x1conv=True, strides=2)
print(blk_down(torch.randn(4, 3, 32, 32)).shape)


torch.Size([4, 3, 32, 32])
torch.Size([4, 16, 16, 16])


## 5 样式迁移与微调

### 5.1 理论题：Fine-tuning

Fine-tuning 通常先加载在大规模数据集（如 ImageNet）上预训练好的模型，再针对自己的任务继续训练。

**1. 哪些层固定，哪些层调整学习率？**

如果目标数据集较小，通常固定前面的卷积层，因为这些层学到的是边缘、纹理、颜色等通用特征；重点训练最后的分类层。如果目标数据集和 ImageNet 差异较大，可以解冻后面几层卷积层，让高层语义特征适应新任务。

学习率一般分层设置：预训练层使用较小学习率，最后新加的输出层使用较大学习率。这样既能保留预训练模型中有用的特征，又能让分类头更快适应新类别。

**2. 采用什么训练策略？**

一种常见策略是：先冻结特征提取部分，只训练新分类层；等分类层基本收敛后，再解冻靠后的若干层，用较小学习率整体微调。训练过程中可以使用数据增强、权重衰减和早停来减少过拟合。


### 5.2 编程题：构造数据增强 Pipeline

题目要求：随机裁剪，比例在 0.08 到 1.0 之间，并缩放到 $224\times 224$；以 50% 概率水平翻转；亮度、对比度、饱和度扰动设为 0.5；最后转为 PyTorch Tensor。


In [6]:
train_augmentation = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.08, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5),
    transforms.ToTensor(),
])

print(train_augmentation)


Compose(
    RandomResizedCrop(size=(224, 224), scale=(0.08, 1.0), ratio=(0.75, 1.3333), interpolation=bilinear, antialias=True)
    RandomHorizontalFlip(p=0.5)
    ColorJitter(brightness=(0.5, 1.5), contrast=(0.5, 1.5), saturation=(0.5, 1.5), hue=None)
    ToTensor()
)


## 6 锚框与标签平滑

### 6.1 理论题：IoU 计算

真实框：

$$
A=[10,10,50,50]
$$

预测框：

$$
B=[30,30,70,70]
$$

两个框的交集左上角为 $(\max(10,30),\max(10,30))=(30,30)$，右下角为 $(\min(50,70),\min(50,70))=(50,50)$。交集宽和高都是 20，所以交集面积为

$$
20\times 20=400.
$$

框 $A$ 面积：

$$
(50-10)(50-10)=1600.
$$

框 $B$ 面积：

$$
(70-30)(70-30)=1600.
$$

并集面积为

$$
1600+1600-400=2800.
$$

因此 IoU 为

$$
IoU=\frac{400}{2800}=\frac17\approx 0.1429.
$$


In [7]:
def box_iou(box_a, box_b):
    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b
    inter_w = max(0, min(ax2, bx2) - max(ax1, bx1))
    inter_h = max(0, min(ay2, by2) - max(ay1, by1))
    inter = inter_w * inter_h
    area_a = max(0, ax2 - ax1) * max(0, ay2 - ay1)
    area_b = max(0, bx2 - bx1) * max(0, by2 - by1)
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0


A = [10, 10, 50, 50]
B = [30, 30, 70, 70]
print(box_iou(A, B))


0.14285714285714285


### 6.2 编程题：Label Smoothing

Label Smoothing 会把 one-hot 标签中原本为 1 的位置改成 $1-\epsilon$，把原本为 0 的位置改成 $\epsilon/(K-1)$。这样模型不会被鼓励过度自信，有助于提升泛化能力。


In [9]:
def label_smoothing(labels, num_classes, epsilon=0.1):
    if labels.dim() != 1:
        labels = labels.reshape(-1)
    off_value = epsilon / (num_classes - 1)
    on_value = 1.0 - epsilon
    y = torch.full(
        (labels.shape[0], num_classes),
        fill_value=off_value,
        dtype=torch.float32,
        device=labels.device,
    )
    y.scatter_(1, labels.reshape(-1, 1), on_value)
    return y


labels = torch.tensor([0, 2, 1])
smoothed = label_smoothing(labels, num_classes=4, epsilon=0.1)
print(smoothed)
print("row sums:", smoothed.sum(dim=1))


tensor([[0.9000, 0.0333, 0.0333, 0.0333],
        [0.0333, 0.0333, 0.9000, 0.0333],
        [0.0333, 0.9000, 0.0333, 0.0333]])
row sums: tensor([1., 1., 1.])


小结：本次作业主要围绕 CNN 基础模块展开。卷积和池化负责提取局部空间特征，VGG 和 NiN 展示了不同的卷积结构设计思路，Batch Normalization 和残差连接改善了深层网络训练，数据增强与微调则是提高实际任务效果的常用方法。
